# 05. Inspect source-labeled games and illustrate simulation

This notebook has two learning goals: inspect retrospective Elo for rows labeled `WLDs` in 2026, and show how random games produce a series probability. It reads notebook 02's canonical global feature export. Source labels are not independently verified tournament identities.

The simulation uses an explicitly assumed constant probability, independent games, and a seeded generator. It supplies no official participants or bracket and makes no frozen pre-tournament forecast. Run it after notebook 02; it does not depend on notebook 04.


## 01. Locate the project

Import data tools and find the project root. Use the same processed-data folder whether Jupyter starts from the project root or the notebook folder.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

base = Path.cwd()
if not (base / "data" / "processed").is_dir():
    base = base.parent


## 02. Read canonical global Elo features

Load `elo_features_all_years.csv`, parse dates, sort by timestamp and match ID, and check unique games. This uses the maintained export rather than older exploratory Elo files. Inspect the shape and date range before filtering.


In [2]:
df = pd.read_csv(
    base / "data" / "processed" / "elo_features_all_years.csv",
    parse_dates=["date"],
)
df = df.sort_values(["date", "match_id"]).reset_index(drop=True)
assert df["match_id"].is_unique
print(df.shape)
print(df["date"].min(), df["date"].max())


(96075, 10)
2017-01-10 15:44:06+00:00 2026-10-26 21:54:37+00:00


## 03. Select source-labeled WLDs games in 2026

Filter the supplied league label and year, then report games, date range, and the union of both opponent columns. Counting both sides avoids missing teams appearing only as B. This retrospective data slice is not a verified official participant list.


In [3]:
# This describes source labels, not independently verified tournament identities.
worlds_2026 = df.loc[
    df["league"].eq("WLDs") & df["date"].dt.year.eq(2026)
].copy()
print("Source-labeled WLDs games in 2026:", len(worlds_2026))
print("First match:", worlds_2026["date"].min())
print("Last match:", worlds_2026["date"].max())
print("Teams:", pd.unique(worlds_2026[["team_a", "team_b"]].to_numpy().ravel()).size)


Source-labeled WLDs games in 2026: 63
First match: 2026-09-17 09:11:42+00:00
Last match: 2026-10-26 21:54:37+00:00
Teams: 21


## 04. Recover each prior expected probability

Convert pre-match Elo difference into A's probability using scale=400. This probability uses information recorded before that game's outcome. It is used below to reconstruct rating transfers.


In [4]:
worlds_2026 = worlds_2026.copy()

worlds_2026["expected_a"] = (
    1 / (1 + 10 ** (-worlds_2026["elo_diff"] / 400))
)

## 05. Reconstruct post-timestamp ratings

With K=24, calculate each game's equal-and-opposite transfer and sum by team and timestamp before updating ratings. This respects simultaneous selected games. The reconstruction sums only this WLDs slice: another appearance outside the slice at the same timestamp would also be needed to reconstruct the team's full global state.


In [5]:
# Sum simultaneous game transfers before reporting a team's post-timestamp Elo.
K = 24.0
transfers = worlds_2026[["date", "team_a", "team_b", "result_a", "expected_a"]].copy()
transfers["delta"] = K * (transfers["result_a"] - transfers["expected_a"])
team_transfers = pd.concat([
    transfers[["date", "team_a", "delta"]].rename(columns={"team_a": "team"}),
    transfers[["date", "team_b", "delta"]].rename(columns={"team_b": "team"}).assign(delta=lambda x: -x["delta"]),
])
batch_changes = team_transfers.groupby(["date", "team"])["delta"].sum()
for side in ["a", "b"]:
    keys = pd.MultiIndex.from_arrays([worlds_2026["date"], worlds_2026[f"team_{side}"]])
    worlds_2026[f"post_elo_{side}"] = (
        worlds_2026[f"pre_elo_{side}"].to_numpy() + batch_changes.reindex(keys).to_numpy()
    )


## 06. Combine both opponents into a team table

Convert the A and B rating columns into the common columns `team` and `elo`, then concatenate them. Each team can now be tracked regardless of its alphabetical A/B label.


In [6]:
ratings_a = worlds_2026[
    ["date", "team_a", "post_elo_a"]
].rename(columns={
    "team_a": "team",
    "post_elo_a": "elo"
})

ratings_b = worlds_2026[
    ["date", "team_b", "post_elo_b"]
].rename(columns={
    "team_b": "team",
    "post_elo_b": "elo"
})

all_ratings = pd.concat(
    [ratings_a, ratings_b],
    ignore_index=True
)

## 07. Inspect the reconstructed changes

Display prior and post-timestamp ratings beside dates and opponents. This is a diagnostic view of selected historical records. It is not an evaluation score or a prediction of future results.


In [7]:
# Display results
print(
    worlds_2026[
        [
            "date",
            "team_a",
            "team_b",
            "pre_elo_a",
            "pre_elo_b",
            "post_elo_a",
            "post_elo_b",
        ]
    ].head()
)

                           date           team_a       team_b    pre_elo_a  \
95085 2026-09-17 09:11:42+00:00  Invictus Gaming  Top Esports  1766.305282   
95086 2026-09-17 10:14:41+00:00  Invictus Gaming  Top Esports  1777.067350   
95087 2026-09-17 11:11:25+00:00  Invictus Gaming  Top Esports  1763.099511   
95088 2026-09-17 12:10:40+00:00  Invictus Gaming  Top Esports  1750.081062   
95105 2026-09-18 09:12:27+00:00        JD Gaming      Team WE  1709.536272   

         pre_elo_b   post_elo_a   post_elo_b  
95085  1730.335523  1777.067350  1719.573455  
95086  1719.573455  1763.099511  1733.541294  
95087  1733.541294  1750.081062  1746.559743  
95088  1746.559743  1737.959444  1758.681360  
95105  1710.684194  1697.575919  1722.644547  


## 08. Rank the latest observed ratings in the slice

Keep each team's latest selected observation and sort by Elo. Teams can have different observation dates; the ratings were not frozen before the selected games. The resulting table is not a simulated tournament ranking.


In [8]:
latest_ratings = (
    all_ratings
    .sort_values("date")
    .drop_duplicates(subset="team", keep="last")
    .sort_values("elo", ascending=False)
    .reset_index(drop=True)
)

latest_ratings["rank"] = latest_ratings.index + 1

print(latest_ratings[["rank", "team", "elo", "date"]].to_string(index=False))

 rank                team         elo                      date
    1               Gen.G 1974.697651 2026-10-25 22:43:24+00:00
    2     Bilibili Gaming 1908.462944 2026-10-26 18:03:09+00:00
    3          G2 Esports 1870.203107 2026-10-25 18:11:29+00:00
    4           Dplus Kia 1867.970408 2026-10-25 20:10:09+00:00
    5 Hanwha Life Esports 1867.730462 2026-10-25 17:19:41+00:00
    6                  T1 1839.362360 2026-10-26 17:12:02+00:00
    7        Karmine Corp 1838.488355 2026-10-26 18:03:09+00:00
    8                 LØS 1832.870642 2026-10-26 20:01:50+00:00
    9  Team Secret Whales 1826.312433 2026-10-26 20:01:50+00:00
   10         Team Liquid 1823.581212 2026-10-26 21:54:37+00:00
   11                LYON 1816.891497 2026-10-25 22:43:24+00:00
   12     Invictus Gaming 1784.344376 2026-10-25 17:19:41+00:00
   13         Top Esports 1781.102631 2026-10-25 18:11:29+00:00
   14     Anyone's Legend 1778.186132 2026-10-26 21:54:37+00:00
   15        Movistar KOI 1749.018392 20

## 09. Set the illustrative probability and random seed

Seed the random generator and assume a single-game probability of 0.3257. This number is an example input rather than a fitted matchup prediction. The following games are independent and keep the same probability throughout a series.


In [9]:
import random
random.seed(2026)
ILLUSTRATIVE_GAME_PROBABILITY = 0.3257  # Assumed example, not an estimated matchup.


## 10. Simulate one game

Draw uniformly between zero and one and return an A win when the draw is below the assumed probability. Across many calls, A's win frequency approaches that input probability.


In [10]:
def simulate_game(p):
    r = random.random()
    if r < p:
        return True
    else:
        return False

## 11. Estimate a game win rate

Run 10,000 simulated games and divide A's wins by the simulation count. Monte Carlo sampling introduces variation even with a fixed underlying probability. The seed makes this particular sequence reproducible.


In [11]:
n_simulations = 10000
team_a_wins = 0

for _ in range(n_simulations):
    team_a_win = simulate_game(ILLUSTRATIVE_GAME_PROBABILITY)
    if team_a_win:
        team_a_wins += 1


print("Estimated win probability:", team_a_wins / n_simulations)

Estimated win probability: 0.3293


## 12. Simulate a best-of-five series

Keep drawing games until a team reaches three wins and return whether A won the series. The assumed game probability does not change with side, adaptation, fatigue, or results. These are illustrative mechanics, not asserted official tournament rules.


In [12]:
def simulate_bo5(p):
    team_a_wins = 0
    team_b_wins = 0
    while team_a_wins < 3 and team_b_wins < 3:
        result = simulate_game(p)
        if result:
            team_a_wins += 1
        else:
            team_b_wins += 1 
    if team_a_wins == 3:
        return True 
    else:
        return False

## 13. Estimate a series win probability

Repeat the series simulation 10,000 times and calculate the fraction won by A. A series probability differs from the single-game probability because three wins are required. No bracket or tournament advancement is simulated.


In [13]:
team_a_wins_bo5 = 0

for _ in range(n_simulations):
    result = simulate_bo5(ILLUSTRATIVE_GAME_PROBABILITY)
    if result:
        team_a_wins_bo5 += 1
    
print("Estimated win probability bo5:", team_a_wins_bo5 / n_simulations)


Estimated win probability bo5: 0.2009


## 14. Check against exact probability calculations

For fixed independent games, winning a best-of-five has the same probability as winning at least three of five games in a binomial calculation. Compare game and series simulation frequencies with their exact values using five Monte Carlo standard errors as tolerance. This checks numerical mechanics, not predictive accuracy of the assumed input.


In [14]:
from math import comb
p_game = ILLUSTRATIVE_GAME_PROBABILITY
exact_bo5 = sum(comb(5, wins) * p_game ** wins * (1 - p_game) ** (5 - wins) for wins in range(3, 6))
simulated_game = team_a_wins / n_simulations
simulated_bo5 = team_a_wins_bo5 / n_simulations
assert abs(simulated_game - p_game) < 5 * np.sqrt(p_game * (1 - p_game) / n_simulations)
assert abs(simulated_bo5 - exact_bo5) < 5 * np.sqrt(exact_bo5 * (1 - exact_bo5) / n_simulations)
print("Assumed game probability:", p_game)
print("Exact illustrative best-of-five probability:", exact_bo5)
print("Monte Carlo mechanics checks passed.")

Assumed game probability: 0.3257
Exact illustrative best-of-five probability: 0.19869885064525294
Monte Carlo mechanics checks passed.
